# 02 · Episodes, states, labels and panel

**Phase:** data (plan §7.3, step 2). **Looks at outcomes vs predictors?** No: labels are built from ERT only, and the panel only attaches raw indicators.

Outputs (all in `data/processed/`):

| File | One row per | Content |
|---|---|---|
| `democratic_episodes.parquet` | democratic episode (`aut_ep_prch == 1`) | onset, confirmation, breakdown years; outcome; severity |
| `country_year_states.parquet` | country-year | stage 0 / 1 / 2 under the configured state source; `pop_group` for the training populations (D-24) |
| `labels.parquet` | country × origin year × stage × horizon × event | `y`, `observable`; stage-0 rows = non-democracy training rows for P1/P2 (`declaration_any`) |
| `panel.parquet` | country-year | identifiers, raw indicators from `config/features.yaml`, stage |

Definitions: plan §3.2–3.5 and the docstrings in `src/flypredictor/data/`.

In [1]:
from flypredictor import load_config, Paths

cfg = load_config()
paths = Paths.from_root().ensure()
print("project root:", paths.root)
import pandas as pd
from flypredictor.data import episodes, states, labels, ert_jobs, panel as pnl

print("state source:", cfg["labels"]["state_source"], "| stage-1 event:", cfg["labels"]["stage1_event"],
      "| require ERT democracy:", cfg["democracy"]["require_ert_democracy"])

project root: C:\Users\markg\OneDrive\Desktop\DACCS\DACSS 756 - Machine Learning\DACSS_ML_FP
state source: ert_truncated | stage-1 event: confirmation | require ERT democracy: True


## 1 · Democratic episodes

In [2]:
ert = episodes.load_ert(paths)
eps = episodes.democratic_episodes(ert, cfg)
problems = episodes.check_episodes(eps)
assert problems == [], problems
print(len(eps), "democratic episodes")
eps["outcome_label"].value_counts()

101 democratic episodes


outcome_label
democratic breakdown    71
outcome censored        15
averted regression      13
preempted breakdown      1
diminished democracy     1
Name: count, dtype: int64

In [3]:
eps.to_parquet(paths.episodes_parquet, index=False)
eps.sort_values("start_year").tail(12)[
    ["episode_id", "start_year", "confirm_year_rule", "breakdown_year", "end_year", "outcome_label", "severity_total"]
]

,episode_id,start_year,confirm_year_rule,breakdown_year,end_year,outcome_label,severity_total
62,ARM_2020_2025,2020,2022,<NA>,2025,outcome censored,0.182
35,CIV_2020_2025,2020,2024,2020,2025,democratic breakdown,0.140
86,GRC_2020_2025,2020,2021,<NA>,2025,outcome censored,0.123
94,ROU_2021_2025,2021,2022,<NA>,2025,outcome censored,0.200
15,SEN_2021_2025,2021,2023,<NA>,2025,outcome censored,0.095
57,UKR_2021_2025,2021,2022,<NA>,2025,outcome censored,0.166
65,GNB_2022_2022,2022,2022,2022,2022,democratic breakdown,0.158
98,SVN_2023_2025,2023,2025,<NA>,2025,outcome censored,0.116
96,SVK_2023_2025,2023,2025,<NA>,2025,outcome censored,0.104
69,MDA_2023_2025,2023,2024,<NA>,2025,outcome censored,0.207


## 2 · Real-time states and events

In [4]:
use_rt = cfg["labels"]["state_source"] == "ert_truncated"
rt = ert_jobs.load_realtime(paths) if use_rt else None
ev = episodes.country_events(eps, rt, ert)
st = states.assign_states(ert, eps, cfg, rt)
ev = episodes.split_declarations(ev, st)   # gradual vs abrupt declarations (H1b, decision D-07b)
st.to_parquet(paths.states_parquet, index=False)
print(ev["event"].value_counts().to_dict())
st["stage"].value_counts().sort_index().rename({0: "0 not democracy", 1: "1 onset risk", 2: "2 in known episode"})

{'declaration_any': 262, 'row_breakdown': 104, 'onset': 101, 'confirmation_rule': 101, 'declaration': 101, 'breakdown': 71, 'declaration_abrupt': 54, 'declaration_dem': 47}


stage
0 not democracy       14678
1 onset risk           5031
2 in known episode      148
Name: count, dtype: int64

## 3 · Labels

In [5]:
lab = labels.build_labels(st, ev, cfg)
lab.to_parquet(paths.labels_parquet, index=False)
print(f"{len(lab):,} label rows, {int(lab['observable'].sum()):,} observable")
labels.label_summary(lab, cfg).query("period != 'after_locked'")

193,864 label rows, 181,867 observable


,stage,event,k,horizon_set,period,rows,positives,countries,base_rate
0,0,declaration_any,1,confirmatory,development,12009,135,179,0.011242
1,0,declaration_any,1,confirmatory,locked,1438,38,97,0.026426
2,0,declaration_any,2,confirmatory,development,11992,264,179,0.022015
3,0,declaration_any,2,confirmatory,locked,1378,72,97,0.052250
4,0,declaration_any,3,confirmatory,development,11978,395,179,0.032977
...,...,...,...,...,...,...,...,...,...
61,2,breakdown,1,confirmatory,locked,53,7,17,0.132075
62,2,breakdown,2,confirmatory,development,21,10,7,0.476190
63,2,breakdown,2,confirmatory,locked,48,12,17,0.250000
64,2,breakdown,3,confirmatory,development,21,14,7,0.666667


### Training populations (decision D-24)

Democracy rows keep the H1 label in every population; P1 and P2 add non-democracy rows (not inside a real-time known autocratization episode of any kind) labelled `declaration_any`. Outcome-only counts at *k* = 1 (the full table is audit A10 in notebook 03).

In [6]:
from flypredictor.audit.definitions import population_counts

pc = population_counts(lab, st, cfg)
(pc[(pc["k"] == 1) & (pc["period"] != "after_locked")]
   .pivot_table(index=["population", "pop_group"], columns="period", values=["rows", "positives"], aggfunc="sum"))

positives               rows       
period                         development locked development locked
population pop_group                                                
P1         closed_autocracy             30      2        8691    448
           democracy                    36     53        3255   1707
           electoral_autocracy         105     36        3318    990
P2         democracy                    36     53        3255   1707
           electoral_autocracy         105     36        3318    990
P3         democracy                    36     53        3255   1707

## 4 · Panel (raw indicators + stage)

In [7]:
panel = pnl.build_panel(paths, cfg, st)
panel.to_parquet(paths.panel_parquet, index=False)
print(panel.shape)
panel[panel["stage"] > 0].groupby("stage").size()

(19857, 50)


stage
1    5031
2     148
dtype: int64

## 5 · Manifest of processed files

In [8]:
from flypredictor.manifest import file_entry, write_manifest

files = [paths.episodes_parquet, paths.states_parquet, paths.labels_parquet, paths.panel_parquet]
write_manifest("processed_v16", [file_entry(f, paths.root) for f in files], paths.manifests,
               state_source=cfg["labels"]["state_source"], stage1_event=cfg["labels"]["stage1_event"])
print("now run:  pytest -m realdata")

now run:  pytest -m realdata
